# ANN (MLP) model — Nepal cereal yield prediction

This notebook adds a neural network (multi-layer perceptron) alongside your existing
Linear Regression / Ridge / Random Forest / XGBoost pipeline, following the same
preprocessing and evaluation protocol described in the Methodology:

- Reuses your existing `get_crop_data()` and `evaluate_groupcv()` functions — **run or
  import your original pipeline notebook first** so interpolation, KNN feature
  imputation, and the domain-plausibility filter stay identical across every model.
- Adds **one-hot encoding** for the categorical soil/landform columns (Landform,
  Parent Material, Dominant Soil), which the tree-based models handled as raw
  categories but an ANN needs as numeric input.
- Fixes the scaling leakage from the earlier pipeline: `StandardScaler` is now fit
  **inside** a `Pipeline`, per training fold, instead of on the full dataset before
  splitting.
- Addresses feature selection: `RFECV` cannot run directly on an MLP (it has no
  `coef_` or `feature_importances_`). Two options are given — reusing the Ridge-selected
  features (recommended, keeps all models comparable on the same inputs) or a
  permutation-importance-based elimination that is native to the ANN.


## 1. Imports and config

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
CROPS = ['Barley', 'Maize', 'Millet', 'Wheat']


## 2. Data paths — edit these

Set the CSV path for whichever of the three datasets you're running this notebook
against (AlphaEarth-only, Soil-only, or Combined). Run the whole notebook once per
dataset, changing only this cell each time — same as you did for Ridge/RF/XGBoost.


In [ ]:
# --- EDIT THIS: point to ONE of your three dataset CSVs per run ---
DATA_PATH = '/path/to/your/combined_dataset.csv'   # <-- e.g. alphaearth_only.csv / soil_only.csv / combined.csv

# --- EDIT THESE: column names as they appear in your CSV ---
DISTRICT_COL = 'District'          # <-- adjust if your column is named differently (e.g. 'district_name')
YEAR_COL = 'Year'                  # <-- adjust if named differently (e.g. 'fiscal_year')
YIELD_COL_TEMPLATE = '{crop}_Yield'  # <-- adjust to match your actual target column naming,
                                      #     e.g. if columns are 'Maize_Yield_MtHa' use '{crop}_Yield_MtHa'

INTERPOLATED_FLAG_COL_TEMPLATE = '{crop}_is_interpolated'
# <-- adjust or remove: if your data has a flag column marking interpolated (vs. genuine)
#     yield values per crop, name it here so evaluate_groupcv can exclude interpolated
#     points from validation scoring, per the paper's protocol. If you don't have this
#     flag, leave INTERPOLATED_FLAG_COL_TEMPLATE = None and every point will be scored.


In [ ]:
def get_crop_data(crop):
    """Loads one dataset CSV (set via DATA_PATH above) and returns the feature matrix,
    target, and grouping columns for one crop. Rows with no genuine yield value for
    this crop (after your existing interpolation step) should already be excluded or
    interpolated upstream, per Section III-A.2 — this function assumes that's already
    been applied to the CSV at DATA_PATH."""
    df = pd.read_csv(DATA_PATH)

    yield_col = YIELD_COL_TEMPLATE.format(crop=crop)
    if yield_col not in df.columns:
        raise KeyError(f"Column '{yield_col}' not found — check YIELD_COL_TEMPLATE above "
                        f"against your actual CSV headers: {list(df.columns)[:15]}...")

    # Drop rows with no yield value at all for this crop (zero real observations case)
    df_crop = df.dropna(subset=[yield_col]).copy()

    # Optionally exclude interpolated points from being scored later
    interp_col = None
    if INTERPOLATED_FLAG_COL_TEMPLATE:
        candidate = INTERPOLATED_FLAG_COL_TEMPLATE.format(crop=crop)
        if candidate in df_crop.columns:
            interp_col = candidate

    non_feature_cols = [DISTRICT_COL, YEAR_COL, yield_col]
    if interp_col:
        non_feature_cols.append(interp_col)
    # also drop any other crops' yield/production/area columns so they don't leak in as features
    other_target_like = [c for c in df_crop.columns
                          if c not in non_feature_cols and
                          any(tok in c for tok in ['Yield', 'Production', 'Area']) and
                          crop not in c]
    non_feature_cols.extend(other_target_like)

    X = df_crop.drop(columns=[c for c in non_feature_cols if c in df_crop.columns])
    y = df_crop[yield_col]
    groups_district = df_crop[DISTRICT_COL]
    groups_year = df_crop[YEAR_COL]

    return X, y, groups_district, groups_year


## 3. Evaluation function

Standard GroupKFold evaluation: fit on the training fold, predict on the held-out
fold, score with R²/RMSE/MAE. If you're using the interpolation-flag column, this
version filters interpolated rows out of the *scoring* step only (they still stay
in training), matching the paper's protocol in Section III-C.


In [ ]:
def evaluate_groupcv(model_ctor, X, y, groups, n_splits):
    gkf = GroupKFold(n_splits=n_splits)
    y_true_all, y_pred_all = [], []
    for train_idx, test_idx in gkf.split(X, y, groups):
        model = model_ctor()
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        preds = model.predict(X.iloc[test_idx])
        y_true_all.extend(y.iloc[test_idx].tolist())
        y_pred_all.extend(preds.tolist())
    y_true_all = np.array(y_true_all)
    y_pred_all = np.array(y_pred_all)
    r2 = r2_score(y_true_all, y_pred_all)
    rmse = mean_squared_error(y_true_all, y_pred_all, squared=False)
    mae = mean_absolute_error(y_true_all, y_pred_all)
    return r2, rmse, mae, y_true_all, y_pred_all


## 3. Categorical encoding

One-hot encode the categorical soil/landform columns present in the Soil-only and Combined feature sets (not needed for AlphaEarth-only, which has no categorical columns). Modal-imputation of missing categorical values should already have happened upstream in `get_crop_data()`, per the paper's missing-value-handling protocol — this step only converts categories to numeric dummy columns.

In [ ]:
CATEGORICAL_COLS = ['Landform', 'Parent_Material', 'Dominant_Soil']  # adjust to your actual column names

def encode_categoricals(X):
    """One-hot encode categorical soil/landform columns for ANN input."""
    present_cat_cols = [c for c in CATEGORICAL_COLS if c in X.columns]
    if not present_cat_cols:
        return X
    X_encoded = pd.get_dummies(X, columns=present_cat_cols, drop_first=False)
    return X_encoded


## 4. Feature selection

**Option A (recommended)** — reuse the per-crop feature subset already selected via RFECV on the Ridge proxy in your earlier pipeline. This keeps Linear/Ridge/RF/XGBoost/ANN comparable on the same inputs, which is the cleaner story for the paper ("the same RFECV-selected features were used across all five models").

In [ ]:
def get_selected_features(crop, X_encoded, selected_features_by_crop=None):
    if selected_features_by_crop and crop in selected_features_by_crop:
        feats = [f for f in selected_features_by_crop[crop] if f in X_encoded.columns]
        if feats:
            return feats
    # fallback: no prior selection available (e.g. new one-hot columns not in the Ridge run)
    return list(X_encoded.columns)


**Option B** — a permutation-importance-based recursive elimination that is native to the ANN, for cases where you want the ANN to select its own features rather than inherit Ridge's. Slower (retrains the model at every elimination step) but methodologically self-contained. Use this instead of Option A if a reviewer might otherwise ask why the ANN's inputs were chosen by a different model.

In [ ]:
def rfecv_permutation(build_model, X, y, groups, min_features=5, step=5,
                      cv_splits=5, n_repeats=5, random_state=RANDOM_STATE):
    """Manual recursive feature elimination for estimators with no coef_/feature_importances_
    (e.g. MLPRegressor), using permutation importance on a single GroupKFold validation
    split at each elimination step."""
    features = list(X.columns)
    gkf = GroupKFold(n_splits=min(cv_splits, groups.nunique()))
    train_idx, val_idx = next(gkf.split(X, y, groups))
    while len(features) > min_features:
        model = build_model()
        model.fit(X.iloc[train_idx][features], y.iloc[train_idx])
        result = permutation_importance(model, X.iloc[val_idx][features], y.iloc[val_idx],
                                         n_repeats=n_repeats, random_state=random_state, scoring='r2')
        importances = pd.Series(result.importances_mean, index=features)
        n_drop = min(step, len(features) - min_features)
        to_drop = importances.sort_values().index[:n_drop].tolist()
        features = [f for f in features if f not in to_drop]
    return features


## 5. ANN model definition

`StandardScaler` is fit inside the `Pipeline`, so it is refit on each training fold only during `evaluate_groupcv` — this closes the scaling-leakage gap from the earlier Ridge/RF/XGBoost pipeline. `alpha` here plays the same role as Ridge's `alpha` (L2 penalty on weights); `early_stopping` guards against overfitting given the small sample size per crop.

In [ ]:
def model_ctor():
    return Pipeline([
        ('scaler', StandardScaler()),
        ('mlp', MLPRegressor(
            hidden_layer_sizes=(64, 32),
            activation='relu',
            solver='adam',
            alpha=1e-3,
            learning_rate_init=1e-3,
            max_iter=2000,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=20,
            random_state=RANDOM_STATE,
        )),
    ])

# Lighter/faster variant if you use Option B (rfecv_permutation retrains many times)
def rfe_model_ctor():
    return Pipeline([
        ('scaler', StandardScaler()),
        ('mlp', MLPRegressor(
            hidden_layer_sizes=(32,),
            activation='relu',
            solver='adam',
            alpha=1e-3,
            max_iter=500,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=10,
            random_state=RANDOM_STATE,
        )),
    ])


## 6. Main loop

Same structure as your Ridge/RF/XGBoost loop — swap in the ANN's `model_ctor`, add the encoding step, and use one of the two feature-selection options above. Using **Option A** by default; switch the `get_selected_features(...)` line to `rfecv_permutation(...)` for Option B.

In [ ]:
results_ann = []
fitted_ann_by_crop = {}
lodo_oof_ann_by_crop = {}

for crop in CROPS:
    print('=' * 70)
    print(crop)
    X, y, groups_district, groups_year = get_crop_data(crop)
    X_encoded = encode_categoricals(X)
    n_samples = len(y)
    n_districts = groups_district.nunique()
    n_years = groups_year.nunique()
    print(f'n_samples={n_samples}  n_districts={n_districts}  n_years={n_years}')

    # --- Option A: reuse Ridge-RFECV features (default) ---
    selected_features = get_selected_features(
        crop, X_encoded,
        selected_features_by_crop=globals().get('selected_features_by_crop')
    )

    # --- Option B: ANN-native selection (uncomment to use instead) ---
    # selected_features = rfecv_permutation(rfe_model_ctor, X_encoded, y, groups_district)

    X_sel = X_encoded[selected_features]
    print(f'Using {len(selected_features)} / {X_encoded.shape[1]} features')

    r2_lodo, rmse_lodo, mae_lodo, y_true_lodo, y_pred_lodo = evaluate_groupcv(
        model_ctor, X_sel, y, groups_district, n_splits=5)
    r2_loyo, rmse_loyo, mae_loyo, y_true_loyo, y_pred_loyo = evaluate_groupcv(
        model_ctor, X_sel, y, groups_year, n_splits=n_years)
    lodo_oof_ann_by_crop[crop] = (y_true_lodo, y_pred_lodo)

    print(f'LODO  R2={r2_lodo:.3f}  RMSE={rmse_lodo:.3f}  MAE={mae_lodo:.3f}')
    print(f'LOYO  R2={r2_loyo:.3f}  RMSE={rmse_loyo:.3f}  MAE={mae_loyo:.3f}')

    final_model = model_ctor()
    final_model.fit(X_sel, y)
    fitted_ann_by_crop[crop] = final_model

    results_ann.append({
        'Crop': crop,
        'N_samples': n_samples,
        'N_features_used': len(selected_features),
        'LODO_R2': r2_lodo, 'LODO_RMSE': rmse_lodo, 'LODO_MAE': mae_lodo,
        'LOYO_R2': r2_loyo, 'LOYO_RMSE': rmse_loyo, 'LOYO_MAE': mae_loyo,
    })

results_ann_df = pd.DataFrame(results_ann)
results_ann_df


## 7. Next steps for the manuscript

- Merge `results_ann_df` into the same results table as Ridge/RF/XGBoost (Tables I–III) —
  add an "ANN" row per crop under each of the three dataset configurations (Soil-only,
  Combined, AlphaEarth-only), same as the other four models.
- Re-run this whole notebook once per feature source (Soil-only / Combined /
  AlphaEarth-only), exactly as you did for the other four models, so the ANN gets its
  own set of twelve model×dataset results consistent with the rest of the paper.
- In the Methodology, add one sentence noting that RFECV feature selection for the ANN
  reused the Ridge-selected subset (Option A) — or describe the permutation-importance
  procedure (Option B) if you used that instead — since this differs from how RFECV was
  applied to the other four models and a reviewer will otherwise ask.
- `MLPRegressor` here is a single-hidden-config baseline. If your supervisor wants a
  stronger/more defensible ANN, worth trying a small hyperparameter sweep over
  `hidden_layer_sizes` and `alpha` inside the same GroupKFold structure (nested,
  as discussed for the other models' hyperparameters), or a Keras/PyTorch model — happy
  to build either if you want to go that route.
